# 🏥 PreventIQ — Illness Risk Prediction Model
### Hackathon-Ready ML Pipeline | State & District Level Predictions
---

In [ ]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from scipy.stats import pointbiserialr, ttest_ind
import joblib
import os
from pathlib import Path
import json

print('✅ All imports successful')

In [ ]:
# ============================================================
# CELL 2: LOAD DATA
# ============================================================
df = pd.read_excel('final_health_data.xlsx')
print(f'Dataset shape: {df.shape}')
print(f'\nTarget distribution:')
print(df['ill'].value_counts())
print(f'\nClass imbalance ratio: {df["ill"].value_counts()[1] / df["ill"].value_counts()[0]:.2f}')
df.head()

In [ ]:
# ============================================================
# CELL 3: PREPROCESSING — SCALE + ENCODE
# ============================================================

# --- Scale numerical features ---
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Remove target and binary indicator columns from scaling
cols_to_scale = [
    'age', 'household_size', 'chronic_disease', 'hypertension', 'diabetes',
    'asthma', 'missed_visits_12m', 'missed_screenings_12m',
    'days_since_last_visit', 'followup_delay_days', 'screening_completed',
    'vaccination_up_to_date', 'facility_access_km', 'region_health_index'
]
cols_to_scale = [c for c in cols_to_scale if c in df.columns]

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

# --- Encode categorical features ---
cat_features = ['age_group', 'locality_type', 'literacy_status', 'income_bracket', 'occupation_type']
cat_present = [c for c in cat_features if c in df_scaled.columns]
df_scaled = pd.get_dummies(df_scaled, columns=cat_present, drop_first=False)

print(f'Scaled {len(cols_to_scale)} numerical features')
print(f'Encoded {len(cat_present)} categorical features')
print(f'Final shape: {df_scaled.shape}')

In [ ]:
# ============================================================
# CELL 4: FEATURE SELECTION VIA STATISTICAL TESTS
# ============================================================

target = df_scaled['ill']
all_num_features = df_scaled.select_dtypes(include=[np.number]).columns.tolist()
all_num_features = [f for f in all_num_features if f != 'ill']

t_test_results = []
for feature in all_num_features:
    ill_group = df_scaled[target == 1][feature].dropna()
    not_ill_group = df_scaled[target == 0][feature].dropna()
    t_stat, p_value = ttest_ind(ill_group, not_ill_group)
    mean_diff = ill_group.mean() - not_ill_group.mean()
    pooled_std = np.sqrt((ill_group.std()**2 + not_ill_group.std()**2) / 2)
    cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0
    t_test_results.append({'Feature': feature, 'T-Stat': t_stat, 'P-Value': p_value, 'Cohens_d': cohens_d})

t_test_df = pd.DataFrame(t_test_results).sort_values('P-Value')

# Select significant features — CRITICAL FIX: remove state-specific binary leakage features
sig_raw = t_test_df[t_test_df['P-Value'] < 0.05]['Feature'].tolist()

# Remove state indicator columns (is_StateName) — these are administrative, not clinical
# Keeping only the locality type indicators (is_City, is_Town, is_Village)
state_is_cols = [f for f in sig_raw if f.startswith('is_') and f not in ['is_City', 'is_Town', 'is_Village']]
significant_features = [f for f in sig_raw if f not in state_is_cols]

print(f'Total significant features (p<0.05): {len(sig_raw)}')
print(f'State-specific columns removed (data leakage risk): {state_is_cols}')
print(f'\nFinal clinical significant features ({len(significant_features)}):')
print(significant_features)

print('\nTop 10 by effect size (Cohen\'s d):')
print(t_test_df[t_test_df['Feature'].isin(significant_features)]
      .sort_values('Cohens_d', key=abs, ascending=False)
      .head(10)[['Feature', 'P-Value', 'Cohens_d']].to_string(index=False))

In [ ]:
# ============================================================
# CELL 5: TRAIN/TEST SPLIT WITH STRATIFICATION
# ============================================================

X = df_scaled[significant_features].fillna(df_scaled[significant_features].mean())
y = df_scaled['ill']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Features:     {X_train.shape[1]}')
print(f'\nClass balance in train: {y_train.value_counts().to_dict()}')

In [ ]:
# ============================================================
# CELL 7a: INSTALL & IMPORT XGBOOST
# ============================================================
import subprocess, sys

try:
    import xgboost as xgb
    print(f'✅ XGBoost already installed — version {xgb.__version__}')
except ImportError:
    print('📦 Installing XGBoost...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
    import xgboost as xgb
    print(f'✅ XGBoost installed — version {xgb.__version__}')

from xgboost import XGBClassifier
print('XGBClassifier ready')

In [ ]:
# ============================================================
# CELL 7b: XGBOOST — FULL TRAINING + EVALUATION + TUNING
# ============================================================
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, RocCurveDisplay
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Calculate class imbalance ratio for scale_pos_weight ──
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f'Class balance → Not-ill: {neg}, Ill: {pos}, scale_pos_weight: {spw:.3f}')

# ── Baseline XGBoost ──────────────────────────────────────
print('\n' + '='*60)
print('STEP 1: BASELINE XGBOOST')
print('='*60)

xgb_base = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,   # handles class imbalance
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_base.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_base  = xgb_base.predict(X_test)
y_prob_base  = xgb_base.predict_proba(X_test)[:, 1]

base_metrics = {
    'Accuracy':  accuracy_score(y_test, y_pred_base),
    'Precision': precision_score(y_test, y_pred_base, zero_division=0),
    'Recall':    recall_score(y_test, y_pred_base, zero_division=0),
    'F1-Score':  f1_score(y_test, y_pred_base, zero_division=0),
    'ROC-AUC':   roc_auc_score(y_test, y_prob_base)
}
print('Baseline XGBoost:')
for k, v in base_metrics.items():
    print(f'  {k:<12}: {v:.4f}')

# ── Hyperparameter Tuning ─────────────────────────────────
print('\n' + '='*60)
print('STEP 2: RANDOMIZED HYPERPARAMETER SEARCH (50 trials)')
print('='*60)

param_grid = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [3, 4, 5, 6, 8],
    'learning_rate':     [0.01, 0.05, 0.1, 0.2],
    'subsample':         [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree':  [0.5, 0.6, 0.7, 0.8, 1.0],
    'min_child_weight':  [1, 3, 5, 7],
    'gamma':             [0, 0.1, 0.2, 0.5],
    'reg_alpha':         [0, 0.01, 0.1, 1],
    'reg_lambda':        [0.5, 1, 2, 5]
}

xgb_search = XGBClassifier(
    scale_pos_weight=spw,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    xgb_search, param_grid,
    n_iter=50,
    scoring='f1',
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)
search.fit(X_train, y_train)

xgb_tuned = search.best_estimator_
print(f'\nBest params: {search.best_params_}')
print(f'Best CV F1:  {search.best_score_:.4f}')

# ── Tuned model evaluation ────────────────────────────────
y_pred_tuned = xgb_tuned.predict(X_test)
y_prob_tuned = xgb_tuned.predict_proba(X_test)[:, 1]

tuned_metrics = {
    'Accuracy':  accuracy_score(y_test, y_pred_tuned),
    'Precision': precision_score(y_test, y_pred_tuned, zero_division=0),
    'Recall':    recall_score(y_test, y_pred_tuned, zero_division=0),
    'F1-Score':  f1_score(y_test, y_pred_tuned, zero_division=0),
    'ROC-AUC':   roc_auc_score(y_test, y_prob_tuned)
}
print('\nTuned XGBoost:')
for k, v in tuned_metrics.items():
    print(f'  {k:<12}: {v:.4f}')

# ── Classification report ─────────────────────────────────
print('\nDetailed Classification Report (Tuned XGBoost):')
print(classification_report(y_test, y_pred_tuned, target_names=['Not Ill', 'Ill']))

# ── Save both models ──────────────────────────────────────
import joblib
from pathlib import Path
models_dir = Path('models')
models_dir.mkdir(exist_ok=True)
joblib.dump(xgb_base,  models_dir / 'xgboost_baseline.joblib')
joblib.dump(xgb_tuned, models_dir / 'xgboost_tuned.joblib')
print('\n✅ XGBoost baseline + tuned models saved')

In [ ]:
# ============================================================
# CELL 7c: XGBOOST VISUALIZATIONS
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import RocCurveDisplay, confusion_matrix

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('PreventIQ — XGBoost Analysis', fontsize=16, fontweight='bold')

# 1. Baseline vs Tuned metrics comparison
ax1 = axes[0, 0]
metric_names = list(base_metrics.keys())
base_vals  = list(base_metrics.values())
tuned_vals = list(tuned_metrics.values())
x = np.arange(len(metric_names))
w = 0.35
ax1.bar(x - w/2, base_vals,  w, label='Baseline', color='#3498db', alpha=0.85)
ax1.bar(x + w/2, tuned_vals, w, label='Tuned',    color='#e74c3c', alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(metric_names, rotation=15)
ax1.set_ylim([0.85, 1.02])
ax1.set_title('Baseline vs Tuned XGBoost', fontweight='bold')
ax1.legend()
ax1.set_ylabel('Score')
for i, (b, t) in enumerate(zip(base_vals, tuned_vals)):
    ax1.text(i - w/2, b + 0.002, f'{b:.3f}', ha='center', fontsize=8)
    ax1.text(i + w/2, t + 0.002, f'{t:.3f}', ha='center', fontsize=8)

# 2. Confusion matrix (tuned)
ax2 = axes[0, 1]
cm = confusion_matrix(y_test, y_pred_tuned)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=ax2,
            xticklabels=['Not Ill', 'Ill'], yticklabels=['Not Ill', 'Ill'])
ax2.set_title('Confusion Matrix (Tuned XGBoost)', fontweight='bold')
ax2.set_xlabel('Predicted'); ax2.set_ylabel('Actual')

# 3. Feature importance (gain)
ax3 = axes[1, 0]
fi = xgb_tuned.get_booster().get_score(importance_type='gain')
fi_series = pd.Series(fi).sort_values(ascending=True).tail(15)
fi_series.plot(kind='barh', ax=ax3, color='#e67e22', alpha=0.85)
ax3.set_title('Top 15 Features — XGBoost Gain', fontweight='bold')
ax3.set_xlabel('Gain')

# 4. ROC Curve
ax4 = axes[1, 1]
RocCurveDisplay.from_predictions(y_test, y_prob_base,  name='XGB Baseline', ax=ax4, color='#3498db')
RocCurveDisplay.from_predictions(y_test, y_prob_tuned, name='XGB Tuned',    ax=ax4, color='#e74c3c')
ax4.set_title('ROC Curve — Baseline vs Tuned', fontweight='bold')
ax4.plot([0,1],[0,1],'k--', alpha=0.4)

plt.tight_layout()
plt.savefig('xgboost_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ XGBoost analysis saved to xgboost_analysis.png')

In [ ]:
# ============================================================
# CELL 7d: COMPARE XGBOOST vs ALL OTHER MODELS
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt

# Add XGBoost to the results_df from the main training cell
xgb_row = pd.DataFrame([{
    'Model': 'XGBoost (Tuned) 🏆',
    'Accuracy':  tuned_metrics['Accuracy'],
    'Precision': tuned_metrics['Precision'],
    'Recall':    tuned_metrics['Recall'],
    'F1-Score':  tuned_metrics['F1-Score'],
    'ROC-AUC':   tuned_metrics['ROC-AUC']
}])

xgb_base_row = pd.DataFrame([{
    'Model': 'XGBoost (Baseline)',
    'Accuracy':  base_metrics['Accuracy'],
    'Precision': base_metrics['Precision'],
    'Recall':    base_metrics['Recall'],
    'F1-Score':  base_metrics['F1-Score'],
    'ROC-AUC':   base_metrics['ROC-AUC']
}])

full_results = pd.concat([results_df, xgb_row, xgb_base_row], ignore_index=True)
full_results = full_results.sort_values('F1-Score', ascending=False).reset_index(drop=True)

print('='*75)
print('FULL MODEL LEADERBOARD (sorted by F1-Score)')
print('='*75)
print(full_results.to_string(index=True))

# Update best model if XGBoost is the winner
overall_best_name = full_results.iloc[0]['Model']
if 'XGBoost (Tuned)' in overall_best_name:
    best_model = xgb_tuned
    best_model_name = 'XGBoost (Tuned)'
    print(f'\n🏆 XGBoost is the new champion! Updating best_model → XGBoost (Tuned)')
else:
    print(f'\n🏆 Best model remains: {overall_best_name}')

# Bar chart comparison
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#e74c3c' if 'XGBoost (Tuned)' in m else ('#f39c12' if 'XGBoost' in m else '#3498db')
          for m in full_results['Model']]
bars = ax.barh(full_results['Model'], full_results['F1-Score'], color=colors, alpha=0.88)
ax.set_xlabel('F1-Score', fontweight='bold', fontsize=12)
ax.set_title('PreventIQ — Full Model Leaderboard (F1-Score)', fontweight='bold', fontsize=14)
ax.set_xlim([0.7, 1.02])
for bar, val in zip(bars, full_results['F1-Score']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('model_leaderboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Leaderboard chart saved to model_leaderboard.png')

In [ ]:
# ============================================================
# CELL 6: MODEL PERSISTENCE UTILITIES
# ============================================================

models_dir = Path('models')
models_dir.mkdir(exist_ok=True)

def save_model(model, model_name):
    path = models_dir / f"{model_name.replace(' ', '_').lower()}.joblib"
    joblib.dump(model, path)
    print(f'  ✅ Saved → {path} ({path.stat().st_size/1024:.1f} KB)')
    return str(path)

def load_model(model_name):
    path = models_dir / f"{model_name.replace(' ', '_').lower()}.joblib"
    if path.exists():
        return joblib.load(path)
    return None

def get_or_train_model(model_name, model_obj, X_tr, y_tr, force_retrain=False):
    cached = load_model(model_name)
    if cached is not None and not force_retrain:
        print(f'  📦 Loaded cached {model_name}')
        return cached
    print(f'  🔧 Training {model_name}...')
    model_obj.fit(X_tr, y_tr)
    save_model(model_obj, model_name)
    return model_obj

print('✅ Model persistence utilities ready')

In [ ]:
# ============================================================
# CELL 7: TRAIN ALL MODELS & EVALUATE
# ============================================================

model_configs = {
    'Logistic Regression':    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'K-Nearest Neighbors':    KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':          DecisionTreeClassifier(random_state=42, max_depth=10),
    'Random Forest':          RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, class_weight='balanced'),
    'Gradient Boosting':      GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5),
    'Support Vector Machine': SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced'),
    'Naive Bayes':            GaussianNB(),
}

model_results = []
trained_models = {}

print('='*70)
print('TRAINING & EVALUATING ALL MODELS')
print('='*70)

for name, clf in model_configs.items():
    # ✅ FIX: Always retrain fresh — ensures feature alignment
    clf.fit(X_train, y_train)
    save_model(clf, name)
    trained_models[name] = clf

    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1] if hasattr(clf, 'predict_proba') else y_pred

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_prob)

    model_results.append({
        'Model': name, 'Accuracy': acc, 'Precision': prec,
        'Recall': rec, 'F1-Score': f1, 'ROC-AUC': auc
    })
    print(f'\n{name}:')
    print(f'  Accuracy={acc:.4f} | Precision={prec:.4f} | Recall={rec:.4f} | F1={f1:.4f} | AUC={auc:.4f}')

results_df = pd.DataFrame(model_results).sort_values('F1-Score', ascending=False).reset_index(drop=True)

# ✅ FIX: Pick best model by F1-Score (better than accuracy for imbalanced data)
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]

print(f'\n' + '='*70)
print(f'🏆 BEST MODEL: {best_model_name} (F1={results_df.iloc[0]["F1-Score"]:.4f}, AUC={results_df.iloc[0]["ROC-AUC"]:.4f})')
print('='*70)
print(results_df.to_string(index=False))

In [ ]:
# ============================================================
# CELL 8: CROSS-VALIDATION ON BEST MODEL
# ============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1  = cross_val_score(best_model, X, y, cv=cv, scoring='f1')
cv_auc = cross_val_score(best_model, X, y, cv=cv, scoring='roc_auc')

print(f'Cross-Validation on {best_model_name} (5-Fold Stratified):')
print(f'  F1:      {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'  ROC-AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

In [ ]:
# ============================================================
# CELL 9: MODEL PERFORMANCE VISUALIZATION
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('PreventIQ — Model Performance Dashboard', fontsize=15, fontweight='bold')

# 1. F1-Score comparison
ax1 = axes[0]
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(results_df))]
bars = ax1.barh(results_df['Model'], results_df['F1-Score'], color=colors)
ax1.set_xlabel('F1-Score', fontweight='bold')
ax1.set_title('F1-Score Comparison', fontweight='bold')
ax1.set_xlim([0, 1.05])
for bar, val in zip(bars, results_df['F1-Score']):
    ax1.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=9)

# 2. ROC-AUC comparison
ax2 = axes[1]
bars2 = ax2.barh(results_df['Model'], results_df['ROC-AUC'], color='#9b59b6')
ax2.set_xlabel('ROC-AUC', fontweight='bold')
ax2.set_title('ROC-AUC Comparison', fontweight='bold')
ax2.set_xlim([0, 1.05])
for bar, val in zip(bars2, results_df['ROC-AUC']):
    ax2.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=9)

# 3. Confusion matrix of best model
ax3 = axes[2]
cm = confusion_matrix(y_test, best_model.predict(X_test))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax3,
            xticklabels=['Not Ill', 'Ill'], yticklabels=['Not Ill', 'Ill'])
ax3.set_xlabel('Predicted', fontweight='bold')
ax3.set_ylabel('Actual', fontweight='bold')
ax3.set_title(f'Confusion Matrix\n({best_model_name})', fontweight='bold')

plt.tight_layout()
plt.savefig('model_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualization saved to model_performance.png')

In [ ]:
# ============================================================
# CELL 10: FEATURE IMPORTANCE
# ============================================================

if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=significant_features)
    fi = fi.sort_values(ascending=True)

    plt.figure(figsize=(10, 6))
    fi.plot(kind='barh', color='#e74c3c')
    plt.xlabel('Feature Importance', fontweight='bold')
    plt.title(f'Feature Importance — {best_model_name}', fontweight='bold')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('\nTop 5 features:')
    print(fi.sort_values(ascending=False).head(5))
elif hasattr(best_model, 'coef_'):
    coef = pd.Series(np.abs(best_model.coef_[0]), index=significant_features).sort_values(ascending=True)
    coef.plot(kind='barh', color='#e74c3c')
    plt.title(f'Feature Coefficients — {best_model_name}', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CELL 11: STATE-WISE RISK PREDICTIONS
# ============================================================

# ✅ KEY FIX: Use model's own feature list to slice data — guarantees alignment
model_feature_list = list(best_model.feature_names_in_) if hasattr(best_model, 'feature_names_in_') else significant_features

print('='*70)
print('STATE-WISE ILLNESS RISK PREDICTIONS')
print('='*70)
print(f'Using model: {best_model_name}')
print(f'Model features ({len(model_feature_list)}): {model_feature_list}\n')

all_states = df_scaled['state_or_ut'].unique()
state_predictions = []

for state in sorted(all_states):
    state_data = df_scaled[df_scaled['state_or_ut'] == state][model_feature_list].copy()
    state_data = state_data.fillna(state_data.mean())

    if len(state_data) > 0:
        risk_probs = best_model.predict_proba(state_data)[:, 1]
        avg_risk = risk_probs.mean()
        state_predictions.append({
            'State': state,
            'Sample_Count': len(state_data),
            'Avg_Risk_Probability': avg_risk,
            'Risk_Percentage': avg_risk * 100,
            'Status': 'HIGH RISK ⚠️' if avg_risk > 0.5 else 'LOW RISK ✅',
            'Max_Risk': risk_probs.max(),
            'Min_Risk': risk_probs.min()
        })

predictions_df = pd.DataFrame(state_predictions).sort_values('Avg_Risk_Probability', ascending=False)
predictions_df = predictions_df.reset_index(drop=True)

print(predictions_df.to_string(index=False))
print(f'\nHigh-risk states: {(predictions_df["Avg_Risk_Probability"] > 0.5).sum()}')
print(f'Low-risk states:  {(predictions_df["Avg_Risk_Probability"] <= 0.5).sum()}')

In [ ]:
# ============================================================
# CELL 12: STATE RISK VISUALIZATION
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('PreventIQ — State-wise Illness Risk Assessment', fontsize=15, fontweight='bold')

# 1. All states horizontal bar
ax1 = axes[0]
plot_df = predictions_df.sort_values('Risk_Percentage', ascending=True)
bar_colors = ['#e74c3c' if v > 50 else '#2ecc71' for v in plot_df['Risk_Percentage']]
bars = ax1.barh(plot_df['State'], plot_df['Risk_Percentage'], color=bar_colors, alpha=0.85)
ax1.axvline(x=50, color='black', linestyle='--', linewidth=1.5, label='Risk Threshold (50%)')
ax1.set_xlabel('Risk Percentage (%)', fontweight='bold')
ax1.set_title('All States — Illness Risk %', fontweight='bold')
ax1.legend()
ax1.set_xlim([0, 105])
for bar, val in zip(bars, plot_df['Risk_Percentage']):
    ax1.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=8)

# 2. Top 10 high risk states
ax2 = axes[1]
top10 = predictions_df.head(10)
ax2.barh(top10['State'][::-1], top10['Risk_Percentage'][::-1], color='#e74c3c', alpha=0.85)
ax2.set_xlabel('Risk Percentage (%)', fontweight='bold')
ax2.set_title('Top 10 Highest-Risk States', fontweight='bold')
ax2.set_xlim([0, 105])
for i, (_, row) in enumerate(top10[::-1].iterrows()):
    ax2.text(row['Risk_Percentage'] + 0.5, i, f'{row["Risk_Percentage"]:.1f}%', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('state_risk.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ State risk chart saved')

In [ ]:
# ============================================================
# CELL 13: DISTRICT-WISE RISK PREDICTIONS
# ============================================================

print('='*70)
print('DISTRICT-WISE ILLNESS RISK PREDICTIONS')
print('='*70)

district_predictions = []

for state in sorted(all_states):
    state_original = df[df['state_or_ut'] == state].copy()
    for district in sorted(state_original['district_name'].unique()):
        district_idx = state_original[state_original['district_name'] == district].index
        # ✅ FIX: Use model_feature_list — perfect alignment every time
        dist_data = df_scaled.loc[district_idx, model_feature_list].copy()
        dist_data = dist_data.fillna(dist_data.mean())

        if len(dist_data) > 0:
            risk_probs = best_model.predict_proba(dist_data)[:, 1]
            avg_risk = risk_probs.mean()
            district_predictions.append({
                'State': state,
                'District_Name': district,
                'Sample_Count': len(dist_data),
                'Avg_Risk_Probability': avg_risk,
                'Risk_Percentage': avg_risk * 100,
                'Status': 'HIGH RISK ⚠️' if avg_risk > 0.5 else 'LOW RISK ✅',
                'Max_Risk': risk_probs.max(),
                'Min_Risk': risk_probs.min()
            })

district_predictions_df = pd.DataFrame(district_predictions).sort_values('Avg_Risk_Probability', ascending=False)
district_predictions_df = district_predictions_df.reset_index(drop=True)

total = len(district_predictions_df)
high_risk = (district_predictions_df['Avg_Risk_Probability'] > 0.5).sum()
print(f'Total districts analyzed: {total}')
print(f'High-risk districts: {high_risk}')
print(f'Low-risk districts:  {total - high_risk}')
print(f'\nTop 20 High-Risk Districts:')
print(district_predictions_df.head(20)[['State','District_Name','Risk_Percentage','Status']].to_string(index=False))

In [ ]:
# ============================================================
# CELL 14: DISTRICT VISUALIZATION
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('PreventIQ — District-level Risk Analysis', fontsize=15, fontweight='bold')

# 1. Top 20 high-risk districts
ax1 = axes[0]
top20_high = district_predictions_df[district_predictions_df['Avg_Risk_Probability'] > 0.5].head(20)
if len(top20_high) > 0:
    labels = [f"{r['District_Name']} ({r['State'][:8]})" for _, r in top20_high.iterrows()]
    ax1.barh(labels[::-1], top20_high['Risk_Percentage'].values[::-1], color='#e74c3c', alpha=0.85)
    ax1.set_xlabel('Risk Percentage (%)', fontweight='bold')
    ax1.set_title('Top 20 High-Risk Districts', fontweight='bold')
    ax1.set_xlim([0, 105])

# 2. Top 20 safest districts
ax2 = axes[1]
top20_safe = district_predictions_df[district_predictions_df['Avg_Risk_Probability'] <= 0.5] \
    .sort_values('Avg_Risk_Probability').head(20)
if len(top20_safe) > 0:
    labels2 = [f"{r['District_Name']} ({r['State'][:8]})" for _, r in top20_safe.iterrows()]
    ax2.barh(labels2[::-1], top20_safe['Risk_Percentage'].values[::-1], color='#2ecc71', alpha=0.85)
    ax2.set_xlabel('Risk Percentage (%)', fontweight='bold')
    ax2.set_title('Top 20 Safest Districts', fontweight='bold')
    ax2.set_xlim([0, 100])

plt.tight_layout()
plt.savefig('district_risk.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ District risk chart saved')

In [ ]:
# ============================================================
# CELL 15: SAVE MODEL METADATA (PRODUCTION READY)
# ============================================================

import datetime

metadata = {
    'model_name': best_model_name,
    'accuracy':   float(results_df.iloc[0]['Accuracy']),
    'f1_score':   float(results_df.iloc[0]['F1-Score']),
    'roc_auc':    float(results_df.iloc[0]['ROC-AUC']),
    'features':   model_feature_list,
    'n_features': len(model_feature_list),
    'training_samples': int(X_train.shape[0]),
    'training_date': datetime.datetime.now().isoformat(),
    'model_version': '2.0'
}

with open(models_dir / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Model metadata saved:')
print(json.dumps(metadata, indent=2))

In [ ]:
# ============================================================
# CELL 16: INTERACTIVE RISK ASSESSMENT WIDGET
# ============================================================

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

output_widget = widgets.Output()

state_dropdown = widgets.Dropdown(
    options=['Select a State...'] + sorted(predictions_df['State'].tolist()),
    description='State:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='350px')
)

district_dropdown = widgets.Dropdown(
    options=['All Districts'],
    description='District:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='350px')
)

def update_districts(change):
    state = change['new']
    if state == 'Select a State...':
        district_dropdown.options = ['All Districts']
    else:
        districts = ['All Districts'] + sorted(
            district_predictions_df[district_predictions_df['State'] == state]['District_Name'].unique().tolist()
        )
        district_dropdown.options = districts

state_dropdown.observe(update_districts, names='value')

def analyze_risk(b):
    with output_widget:
        clear_output(wait=True)
        state = state_dropdown.value
        district = district_dropdown.value

        if state == 'Select a State...':
            print('❌ Please select a State')
            return

        print('\n' + '='*60)
        print('🏥 PREVENTIQ — ILLNESS RISK ASSESSMENT REPORT')
        print('='*60)

        if district == 'All Districts':
            row = predictions_df[predictions_df['State'] == state]
            if len(row) == 0:
                print(f'No data for {state}')
                return
            row = row.iloc[0]
            print(f'📍 Location: {state} (All Districts)')
            print(f'👥 Samples:  {int(row["Sample_Count"])}')
            print(f'⚠️  Risk:     {row["Risk_Percentage"]:.1f}%')
            print(f'📊 Status:   {row["Status"]}')
        else:
            row = district_predictions_df[
                (district_predictions_df['State'] == state) &
                (district_predictions_df['District_Name'] == district)
            ]
            if len(row) == 0:
                print(f'No data for {district}')
                return
            row = row.iloc[0]
            print(f'📍 Location: {district}, {state}')
            print(f'👥 Samples:  {int(row["Sample_Count"])}')
            print(f'⚠️  Risk:     {row["Risk_Percentage"]:.1f}%')
            print(f'📊 Status:   {row["Status"]}')
            print(f'📈 Max Risk: {row["Max_Risk"]*100:.1f}%')
            print(f'📉 Min Risk: {row["Min_Risk"]*100:.1f}%')

        print('='*60)

btn = widgets.Button(description='🔍 Analyze Risk', button_style='danger',
                     layout=widgets.Layout(width='200px', height='40px'))
btn.on_click(analyze_risk)

print('🎯 PreventIQ — Interactive Risk Assessment Tool')
print('Select a state and district below:')
display(widgets.VBox([state_dropdown, district_dropdown, btn, output_widget]))

In [ ]:
# ============================================================
# CELL 17: FINAL SUMMARY
# ============================================================

print('='*70)
print('🏆 PREVENTIQ MODEL — FINAL SUMMARY')
print('='*70)
print(f'Best Model:        {best_model_name}')
print(f'Accuracy:          {results_df.iloc[0]["Accuracy"]:.4f}')
print(f'F1-Score:          {results_df.iloc[0]["F1-Score"]:.4f}')
print(f'ROC-AUC:           {results_df.iloc[0]["ROC-AUC"]:.4f}')
print(f'Features Used:     {len(model_feature_list)}')
print(f'States Analyzed:   {len(predictions_df)}')
print(f'Districts Analyzed:{len(district_predictions_df)}')
print('='*70)
print('Files saved: models/, model_performance.png, state_risk.png, district_risk.png')